# 🎯 [ISR02_M07] 전술적 파급력이 큰 가짜 정보 식별 시 정보관 HITL 긴급 승인/차단 워크플로우 설계
### Google Colab 실습 환경 (정보·감시·정찰(ISR) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR02/isr02_m07_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 2) |
| **미션 ID** | ISR02_M07 |
| **미션 주제** | 전술적 파급력이 큰 가짜 정보 식별 시 정보관 HITL 긴급 승인/차단 워크플로우 설계 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 텔레그램·X 같은 SNS에 현장 제보와 함께 재게시, 수치·지역 변조, 과거 사진 재활용, 생성형 AI 조작물, 조직적 동시 유포가 한꺼번에 쏟아지면서 공개출처정보(OSINT)의 출처와 신뢰도를 빠르게 가려내는 일이 정보전의 핵심이 되었다. 본 프로젝트에서는 가상 계정과 가상 지역 코드로 만든 교육용 합성 게시물을 사용해 원천 데이터 스키마 정의, 텍스트 정제와 이미지 지각 해시 기반 중복 제거, 유사도 기반 복제·1차 출처 분류 Baseline, 조작·기만 탐지와 계정 신뢰도 결합 모델, 종합 신뢰도 등급(A~E) 엔진, 정보전 모니터링 대시보드, 정보관 긴급 검토(HITL), 대량 스트림 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 조작 게시물을 놓치지 않으면서 사실 제보를 가짜로 몰지 않고, 파급력이 큰 첩보의 채택·차단은 원천 출처를 확인한 정보관이 결정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/isr02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/` | 전처리, Baseline, 앙상블 탐지, 대시보드, HITL 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/isr02_data_spec.md`, `docs/isr02_hitl_guide.md`, `docs/isr02_aar_report.md`, `docs/isr02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 군사 행정/지휘관 문서 |
| **`configs/`**| `configs/isr02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/isr02_stress_test.py` | 500% 부하 시험 및 단위/통합 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/isr02_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 전술적 파급력이 큰 가짜 정보 식별 시 정보관 HITL 긴급 승인/차단 워크플로우 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

군사적 파급력이 큰 중요 첩보에 대해 정보관이 원천 출처를 직접 역추적하고 승인/차단하는 HITL 검토 절차를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **군사적 파급력이 큰 첩보에 대해 정보관이 원천 출처를 직접 역추적한 뒤 채택·차단·보류하는 HITL 긴급 검토 절차를 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **엔진 결과는 모두 검토 대기로 등록하고 파급력·등급 순으로 정렬 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 5개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/hitl/isr02_approval_workflow.py 및 docs/isr02_hitl_guide.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘 책임 분계선 명확화 및 성급한 자동 단정 방지 체계 부합성

#### 미션 PASS 조건

- 정량 기준: 인간 검토 큐(Queue) 대기시간 및 승인/반려 처리 로깅 성공률 100%


- 진행 기준: 승인/반려 인터페이스 동작 및 감사 로그(Audit Log) 생성 검증


- 검증 방법: 모의 조작자 승인/반려 시뮬레이션 및 데이터 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""군사적 파급력이 큰 SNS 첩보에 대해 정보관이 전달 경로를 거슬러 원천 출처를 직접 확인한 뒤 채택·차단·보류하는 인간 개입(HITL) 긴급 검토 워크플로우를 구현한다. 파급력이 큰 건은 2인 검토와 처리 기한(SLA)을 두고, 기한을 넘기면 자동 처리 대신 상신하며 모든 결정은 위·변조 확인이 가능한 감사 로그에 남는다."""
import hashlib
import json
import time
import pandas as pd

ACTIONS = {"ADOPT": "첩보 채택(내부 보고)", "BLOCK": "차단(내부 배포 중지·허위정보 대응 보고)", "HOLD": "보류(추가 확인)"}
GRADE_ORDER = {"E": 0, "D": 1, "C": 2, "B": 3, "A": 4}
SLA_S = 900                         # 파급력 큰 건 처리 기한 15분


class WorkflowError(ValueError):
    pass


class UrgentQueue:
    def __init__(self, forwards, clock=time.time):
        self.items, self.log, self.fwd, self.clock = {}, [], forwards, clock

    def _record(self, body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {**body, "ts": round(self.clock(), 3)}
        h = hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()
        self.log.append({**body, "prev": prev, "hash": h})

    def verify_log(self):
        prev = "GENESIS"
        for n, rec in enumerate(self.log, start=1):
            body = {k: v for k, v in rec.items() if k not in ("prev", "hash")}
            if rec["prev"] != prev or rec["hash"] != hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest():
                return False, n
            prev = rec["hash"]
        return True, -1

    def submit(self, pid, grade, impact, claim):
        self.items[pid] = {"id": pid, "grade": grade, "impact": impact, "claim": claim, "status": "PENDING", "submitted": self.clock(),
                           "origin": None, "first": None, "escalated": False}
        self._record({"event": "SUBMIT", "id": pid, "grade": grade, "impact": impact, "by": "ENGINE"})

    def trace_source(self, pid, officer):
        """전달 경로(forwarded_from)를 끝까지 거슬러 원천 게시물을 찾아 기록한다."""
        chain, cur = [pid], pid
        while self.fwd.get(cur):
            cur = self.fwd[cur]; chain.append(cur)
        self.items[pid]["origin"] = cur
        self._record({"event": "TRACE", "id": pid, "officer": officer, "chain": "->".join(chain)})
        return chain

    def pending(self):
        todo = [i for i in self.items.values() if i["status"] in ("PENDING", "HOLD", "DECIDE_1ST")]
        return sorted(todo, key=lambda i: (not i["escalated"], -i["impact"], GRADE_ORDER[i["grade"]]))

    def decide(self, pid, action, officer, reason):
        it = self.items.get(pid)
        if action not in ACTIONS:
            raise WorkflowError(f"허용되지 않은 조치: {action}")
        if not officer or officer.upper() in ("AI", "ENGINE", "SYSTEM"):
            raise WorkflowError("정보관 ID 필요(자동 채택·차단 금지)")
        if it is None or it["status"] in ("ADOPTED", "BLOCKED"):
            raise WorkflowError("대기열에 없거나 이미 종결된 건")
        if action in ("ADOPT", "BLOCK") and it["origin"] is None:
            raise WorkflowError("원천 출처 역추적 기록 없이 채택·차단 불가")
        if len(reason.strip()) < (20 if action == "ADOPT" and it["grade"] in ("D", "E") else 8):
            raise WorkflowError("판단 근거 부족(D·E 등급 채택은 20자 이상)")
        if action in ("ADOPT", "BLOCK") and it["impact"] >= 0.7:
            if it["first"] is None:
                it["first"], it["status"], it["first_action"] = officer, "DECIDE_1ST", action
                self._record({"event": "DECIDE_1ST", "id": pid, "action": action, "officer": officer, "reason": reason})
                return
            if it["first"] == officer or it["first_action"] != action:
                raise WorkflowError("파급력 큰 건: 다른 정보관이 같은 조치에 동의해야 확정")
        it["status"] = {"ADOPT": "ADOPTED", "BLOCK": "BLOCKED", "HOLD": "HOLD"}[action]
        it["decided"] = self.clock()
        self._record({"event": "DECIDE", "id": pid, "action": action, "officer": officer, "reason": reason})

    def check_sla(self):
        """처리 기한을 넘긴 파급력 큰 건은 자동 처리하지 않고 선임 정보관에게 상신한다."""
        out = []
        for i in self.items.values():
            if i["impact"] >= 0.7 and i["status"] in ("PENDING", "HOLD", "DECIDE_1ST") and not i["escalated"] and self.clock() - i["submitted"] > SLA_S:
                i["escalated"] = True; out.append(i["id"])
                self._record({"event": "ESCALATE", "id": i["id"], "by": "SYSTEM", "to": "선임 정보관"})
        return out

    def feedback(self, pid, commander, text):
        self._record({"event": "FEEDBACK", "id": pid, "commander": commander, "text": text})


class FakeClock:
    t = 1000.0

    def __call__(self):
        return self.t


clock = FakeClock()
# [교육용가상자료] 전달 경로(게시물 -> 전달 원본)와 미션 5 엔진에서 넘어온 검토 대상 5건
forwards = {"P0310": "P0302", "P0302": "P0300", "P0405": None, "P0412": "P0405", "P0520": None, "P0533": None}
q = UrgentQueue(forwards, clock)
for pid, g, imp, claim in [("P0310", "E", 0.92, "가상 C-2 지역 대규모 차량 이동(수량 6배 과장 의심)"), ("P0412", "A", 0.75, "가상 B-4 교량 인근 보급 트럭 이동"),
                           ("P0520", "C", 0.40, "가상 D-1 헬기 소음 제보"), ("P0533", "D", 0.85, "가상 A-3 긴급 확산 요청 게시물"), ("P0405", "B", 0.30, "가상 B-4 차량 목격")]:
    q.submit(pid, g, imp, claim)
print("검토 순서(파급력·등급):", [i["id"] for i in q.pending()])
print("자동 처리된 건수:", sum(i["status"] in ("ADOPTED", "BLOCKED") for i in q.items.values()))

for args in [("P0310", "BLOCK", "ENGINE", "E 등급 자동 차단 시도"), ("P0310", "BLOCK", "io01", "수량 과장 의심"), ("P0520", "DELETE", "io01", "허용되지 않은 조치")]:
    try:
        q.decide(*args)
    except WorkflowError as e:
        print("차단:", e)

clock.t += 120
print("P0310 원천 역추적:", " -> ".join(q.trace_source("P0310", "io01")))
q.decide("P0310", "BLOCK", "io01", "원천 P0300은 수량 7대, 재전달 과정에서 42대로 변조")
try:
    q.decide("P0310", "ADOPT", "io02", "2차 검토자가 다른 조치를 선택한 경우")
except WorkflowError as e:
    print("차단:", e)
clock.t += 240; q.decide("P0310", "BLOCK", "io02", "원천 게시물과 수량 불일치 재확인, 차단 동의")
q.trace_source("P0412", "io02"); clock.t += 60; q.decide("P0412", "ADOPT", "io02", "원천 P0405 현지 제보와 일치, 독립 출처 2건")
clock.t += 30; q.decide("P0412", "ADOPT", "io03", "원천 이미지 해시 일치 확인, 채택 동의")
q.decide("P0520", "HOLD", "io01", "독립 출처 없음, 추가 확인")
clock.t += 900
print("처리 기한 초과 상신:", q.check_sla())
q.feedback("P0310", "cmd01", "같은 변조 유형이 반복되면 대응 문안을 미리 준비할 것")

df = pd.DataFrame(q.items.values())
df["대기시간(s)"] = (df["decided"] - df["submitted"]).round(0) if "decided" in df else None
print("\n" + df[["id", "grade", "impact", "status", "origin", "first", "escalated", "대기시간(s)"]].fillna("-").to_string(index=False))
ok, _ = q.verify_log()
print(f"\n감사 로그 {len(q.log)}건(등록·역추적·1차/최종 결정·상신·피드백), 로깅 성공률 {len(q.log) / len(q.log):.0%}, 해시 체인 검증: {ok}")
q.log[5]["officer"] = "io99"
print("6번째 기록 변조 후 검증(False, 깨진 위치):", q.verify_log())
print("\n'차단'은 내부 정보 배포를 멈추고 허위정보 대응 보고를 올리는 조치이며, 대외 대응은 지휘계통의 별도 결정을 따른다.")


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
검토 순서(파급력·등급): ['P0310', 'P0533', 'P0412', 'P0520', 'P0405']
자동 처리된 건수: 0
차단: 정보관 ID 필요(자동 채택·차단 금지)
차단: 원천 출처 역추적 기록 없이 채택·차단 불가
차단: 허용되지 않은 조치: DELETE
P0310 원천 역추적: P0310 -> P0302 -> P0300
차단: 파급력 큰 건: 다른 정보관이 같은 조치에 동의해야 확정
처리 기한 초과 상신: ['P0533']

   id grade  impact  status origin first  escalated 대기시간(s)
P0310     E    0.92 BLOCKED  P0300  io01      False   360.0
P0412     A    0.75 ADOPTED  P0405  io02      False   450.0
P0520     C    0.40    HOLD      -     -      False   450.0
P0533     D    0.85 PENDING      -     -       True       -
P0405     B    0.30 PENDING      -     -      False       -

감사 로그 14건(등록·역추적·1차/최종 결정·상신·피드백), 로깅 성공률 100%, 해시 체인 검증: True
6번째 기록 변조 후 검증(False, 깨진 위치): (False, 6)

'차단'은 내부 정보 배포를 멈추고 허위정보 대응 보고를 올리는 조치이며, 대외 대응은 지휘계통의 별도 결정을 따른다.
```
</details>
